# Nemotron-3-Nano-30B LoRA — NVIDIA Model Reasoning Challenge

**Competition:** [NVIDIA Nemotron Model Reasoning Challenge](https://www.kaggle.com/competitions/nvidia-nemotron-model-reasoning-challenge)  
**Team:** gdataranger  
**Approach:** Supervised fine-tuning (SFT) of Nemotron-3-Nano-30B with LoRA using Gemini-generated chain-of-thought traces  
**Hardware:** NVIDIA GB10 (DGX Spark) — 128 GB unified memory, aarch64  

> Training was performed off-Kaggle on a GB10 machine using a custom Docker image.
> This notebook documents the setup, data pipeline, training config, and results,
> and demonstrates loading the adapter for inference.


## 1. Overview

The competition asks competitors to fine-tune **Nemotron-3-Nano-30B** (a 30B Mamba/attention hybrid) on
problem-answer pairs drawn from Alice's Wonderland–style reasoning puzzles — bit manipulation, ciphers,
unit conversion, and physics. The evaluation metric extracts a final `\\boxed{...}` answer from the model
output and compares it to the ground-truth label.

### Approach

| Step | What | How |
|---|---|---|
| Data | Peer CoT dataset — 9,500 competition prompts with Gemini-2.0-flash CoT traces | `kienngx/nemotron-30b-competition-trainingdata-cot-labels` |
| Training | 1-epoch LoRA (r=32) SFT with TRL `SFTTrainer` | `scripts/train_lora.py` inside Docker on GB10 |
| Submission | LoRA adapter zipped and uploaded to Kaggle | `scripts/package_submission.sh` |

The v0.1-baseline (raw labels only, no CoT) scored **0.57** on the public leaderboard.
The v0.2-cot run adds full reasoning traces to each training example, targeting a higher score.


## 2. Environment and Dependencies

Training ran inside a custom Docker image built from `Dockerfile.gb10-26-01` on a **GB10 (DGX Spark)**
machine running Ubuntu 24.04 aarch64.

### Hardware

| Property | Value |
|---|---|
| Machine | NVIDIA DGX Spark (GB10) |
| Architecture | aarch64 (ARM64) |
| Memory | 128 GB unified (CPU + GPU) |
| GPU | NVIDIA Blackwell GB10 |
| CUDA driver | 580.142 (CUDA 13.2 forward compat) |

### Docker image — `nemotron-gb10:latest`

Built from `Dockerfile.gb10-26-01`:

```bash
bash scripts/build_image.sh 26-01
# → nemotron-gb10-26-01:latest
```

**Base image:** `nvcr.io/nvidia/pytorch:26.01-py3`  
CUDA 13.1 · PyTorch 2.12 (nv26.01) · Python 3.12 · aarch64

**Key packages installed on top:**

| Package | Version | Purpose |
|---|---|---|
| `transformers` | 4.57.3 | Model/tokenizer loading |
| `peft` | 0.14.0 | LoRA via `get_peft_model` |
| `trl` | 0.15.2 | `SFTTrainer` / `SFTConfig` |
| `accelerate` | 1.3.0 | Distributed training backend |
| `bitsandbytes` | source | QLoRA (sm 80–121 compute caps) |
| `causal-conv1d` | 1.6.x | Mamba SSM depedency, source build |
| `mamba-ssm` | latest | Mamba-2 Triton kernels for NemotronH |

**GB10-specific Docker flags** (note: `--gpus all` is not supported on GB10):

```bash
docker run --rm --privileged \
  -e NVIDIA_VISIBLE_DEVICES=all \
  --ipc=host \
  --ulimit memlock=-1 \
  --ulimit stack=67108864 \
  ...
```

**Note on `mamba-ssm`:** Docker `RUN` steps have no GPU access (isolated OCI namespaces), so
`selective_scan_cuda` cannot be compiled at build time. The Dockerfile patches
`selective_scan_interface.py` with a `try/except` around that import. This is safe — Nemotron-H
uses Mamba-2 Triton kernels exclusively and never calls the legacy CUDA path.


In [ ]:
# Install dependencies for running inference inside this Kaggle notebook.
# (Training was done off-Kaggle on a GB10 machine; these are the runtime-only subset.)
#
# Note: pip may print dependency conflict warnings for pre-existing Kaggle base packages
# (bigframes, tpot, gcsfs, fsspec). These are not caused by our install and do not
# affect the packages below — they are safe to ignore.
import subprocess, sys

pkgs = [
    "transformers==5.5.3",   # >= 5.3.0 required: native NemotronH support, no trust_remote_code
    "peft==0.14.0",
    "accelerate==1.3.0",
    "datasets==3.2.0",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts"] + pkgs,
    check=True
)

import transformers, peft, accelerate, datasets
print(f"transformers {transformers.__version__}")
print(f"peft        {peft.__version__}")
print(f"accelerate  {accelerate.__version__}")
print(f"datasets    {datasets.__version__}")
print("All packages ready.")


## 3. Loading Base Model and LoRA Adapter

The final LoRA adapter will be published to Hugging Face Hub after the best run completes.
Set `ADAPTER_REPO` to the published repo ID, or set `USE_HF_ADAPTER = False` and point
`ADAPTER_PATH` at a Kaggle dataset containing the unzipped adapter files.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL_ID = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
ADAPTER_REPO  = "marksusol/nemotron-nano-30b-lora-reasoning"
USE_HF_ADAPTER = True

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

if USE_HF_ADAPTER:
    model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
else:
    ADAPTER_PATH = "/kaggle/input/nemotron-lora-adapter/adapter"
    model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

model.eval()
print("Model loaded.")


## 4. Prompt Template and Inference Demo

All training examples use the same system prompt and expect the final answer inside `\\boxed{}`.
The inference helper below applies the chat template and runs greedy decoding.


In [ ]:
SYSTEM_PROMPT = (
    "You are a careful reasoning model. "
    "Solve the problem step by step and end with Final answer: \\boxed{...}."
)

def generate_answer(problem: str, max_new_tokens: int = 512) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": problem},
    ]
    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template:
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    else:
        prompt = "\n".join(f"{m['role']}: {m['content']}" for m in messages) + "\nassistant:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, temperature=1.0)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


In [ ]:
print(generate_answer("What is 17 + 25?"))


## 5. Data Pipeline

### Source dataset

We used a competition peer's Kaggle dataset instead of training on raw labels alone:

**[kienngx/nemotron-30b-competition-trainingdata-cot-labels](https://www.kaggle.com/datasets/kienngx/nemotron-30b-competition-trainingdata-cot-labels)**

- 9,500 rows covering all competition problem categories
- `generated_cot` column: full reasoning trace produced by **Gemini-2.0-flash**
- `answer` column: ground-truth final answer (no `\\boxed{}` wrapper)
- 213 rows with degenerate CoT (< 20 chars) were filtered out
- Final split: **8,358 train / 929 valid** (90/10, `random.seed(42)`)

### v0.1-baseline comparison

| Version | `response` field | Train rows |
|---|---|---|
| v0.1-baseline | `Final answer: \\boxed{42}` | 8,550 |
| v0.2-cot | `<Gemini CoT trace>\nFinal answer: \\boxed{42}` | 8,358 |

### JSONL row format

Each row in `data/train.jsonl` and `data/valid.jsonl`:

```json
{"id": "7a962e17",
 "prompt": "<competition problem text>",
 "response": "<Gemini CoT reasoning>\nFinal answer: \\boxed{<answer>}",
 "system": "You are a careful reasoning model..."}
```

Token length statistics (estimated, chars/4):
p50 ≈ 248 tokens · p90 ≈ 627 tokens · p99 ≈ 1,911 tokens — all within `max_seq_length=2048`.


In [ ]:
# Example training row
import json

example = {
    "id": "7a962e17",
    "prompt": "In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers...",
    "response": (
        "Let's analyze the pattern bit by bit...\n"
        "After examining all examples, the rule appears to be XOR with mask 10110110.\n"
        "Applying to 00110100: 00110100 XOR 10110110 = 10000010.\n"
        "Final answer: \\boxed{10000010}"
    ),
    "system": "You are a careful reasoning model. Solve the problem step by step and end with Final answer: \\boxed{...}.",
}
print(json.dumps(example, indent=2))


## 6. Training Configuration

Training ran entirely on the GB10 machine via `bash scripts/run_train.sh`, which launches
`scripts/train_lora.py` inside the `nemotron-gb10:latest` Docker container.

### Docker run command

```bash
docker run --rm --privileged \
  -e NVIDIA_VISIBLE_DEVICES=all \
  --ipc=host \
  --ulimit memlock=-1 \
  --ulimit stack=67108864 \
  --user "$(id -u):$(id -g)" \
  -e HF_TOKEN="${HF_TOKEN}" \
  -v "$(pwd)":/workspace \
  -v "$(pwd)/.cache/triton":/home/ubuntu/.triton \
  -w /workspace \
  nemotron-gb10:latest \
  python scripts/train_lora.py \
    --model-id nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16 \
    --train-file data/train.jsonl \
    --valid-file data/valid.jsonl \
    --output-dir output/adapter_YYYYMMDD_HHMMSS \
    --max-seq-length 2048 \
    --batch-size 1 \
    --grad-accum 8 \
    --learning-rate 2e-4 \
    --num-epochs 1 \
    --lora-r 32 \
    --lora-alpha 64 \
    --lora-dropout 0.05
```

### LoRA configuration

| Parameter | Value | Note |
|---|---|---|
| `r` | 32 | Competition max |
| `lora_alpha` | 64 | 2× rank |
| `lora_dropout` | 0.05 | |
| `target_modules` | `all-linear` | All linear layers |
| `task_type` | `CAUSAL_LM` | |

### SFTConfig

| Parameter | Value |
|---|---|
| `per_device_train_batch_size` | 1 |
| `gradient_accumulation_steps` | 8 (effective batch = 8) |
| `learning_rate` | 2e-4 |
| `num_train_epochs` | 1 |
| `bf16` | True |
| `lr_scheduler_type` | cosine |
| `warmup_ratio` | 0.03 |
| `max_seq_length` | 2048 |
| `gradient_checkpointing` | False (NemotronH does not declare support) |

**Why `gradient_checkpointing=False`:** `NemotronHForCausalLM` does not declare
`supports_gradient_checkpointing = True`, so TRL raises a `ValueError` if it is enabled.
The full bf16 model (~60 GB) fits comfortably in 128 GB unified memory without it.


In [ ]:
# Training configuration (for documentation — training was run off-Kaggle)
import torch
from peft import LoraConfig
from trl import SFTConfig

peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)

sft_config = SFTConfig(
    output_dir="./lora-output",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=1.0,
    bf16=True,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,
    max_seq_length=2048,
    gradient_checkpointing=False,  # NemotronH does not declare support
    report_to="none",
)

print("LoRA config:", peft_config)
print("SFT config output_dir:", sft_config.output_dir)


## 7. Evaluation

Local validation runs `scripts/infer_lora.py` to generate predictions on `data/valid.jsonl`,
then `scripts/validate_metric.py` to score them. The metric extracts the last `\\boxed{}`
from the output and compares with numeric tolerance (`rel_tol=1e-4`).

```bash
# Score predictions inside the container
docker run --rm \
  -v "$(pwd)":/workspace \
  -w /workspace \
  nemotron-gb10:latest \
  python scripts/validate_metric.py \
    --predictions output/predictions.jsonl \
    --labels      data/valid_labels.jsonl
```


In [ ]:
import math, re, json

BOXED_RE = re.compile(r"\\boxed\{([^{}]+)\}")

def extract_boxed(text: str) -> str:
    matches = BOXED_RE.findall(text)
    return matches[-1].strip() if matches else ""

def is_correct(pred: str, truth: str, rel_tol: float = 1e-4) -> bool:
    p, t = pred.strip(), truth.strip()
    if p == t:
        return True
    try:
        return math.isclose(float(p), float(t), rel_tol=rel_tol, abs_tol=0.0)
    except Exception:
        return False

# Quick sanity check
assert is_correct("42", "42")
assert is_correct("3.14159", "3.14160", rel_tol=1e-4)
assert not is_correct("42", "43")
print("Metric checks passed.")


## 8. Results and Discussion

| Version | Training data | Val Acc | Kaggle Score | Notes |
|---|---|---|---|---|
| v0.1-baseline | Competition labels only (no CoT) | 43.5% (413/950) | 0.57 | First full run |
| v0.2-cot | Peer CoT dataset (Gemini-2.0-flash) | TBD | TBD | CoT traces in response |

### Key observations (v0.1-baseline)

- Training on bare `Final answer: \\boxed{...}` labels (no reasoning) achieved 0.57 on the public
  leaderboard despite a 43.5% local validation accuracy, suggesting the model learned answer
  formatting but not robust reasoning.
- v0.2-cot targets improvement by providing explicit step-by-step reasoning traces for each
  training example, so the model can learn the intermediate reasoning process rather than
  just the final answer pattern.

*(This section will be updated after v0.2-cot training and Kaggle submission.)*


## 9. Reproducibility Notes

| Artifact | Location |
|---|---|
| Source code | [github.com/msusol/kaggle-nemotron-model-reasoning-challenge](https://github.com/msusol/kaggle-nemotron-model-reasoning-challenge) |
| Docker image | `Dockerfile.gb10-26-01` — base `nvcr.io/nvidia/pytorch:26.01-py3` |
| Training script | `scripts/train_lora.py` |
| Training runner | `scripts/run_train.sh` (reads `configs/nemotron.yaml`) |
| Inference script | `scripts/infer_lora.py` |
| Validation script | `scripts/validate_metric.py` |
| Packaging script | `scripts/package_submission.sh` |
| Data conversion | `scripts/download_peer_cot.py` |
| LoRA adapter (v0.1-baseline) | `output/adapter_20260503_203554` — *to be published to HF Hub* |
| LoRA adapter (v0.2-cot) | `output/adapter_YYYYMMDD_HHMMSS` — *TBD after training completes* |

### Steps to reproduce

```bash
# 1. Clone repo
git clone https://github.com/msusol/kaggle-nemotron-model-reasoning-challenge
cd kaggle-nemotron-model-reasoning-challenge

# 2. Build Docker image (GB10 / DGX Spark)
bash scripts/build_image.sh 26-01

# 3. Download and convert peer CoT dataset
bash scripts/run_download_peer_cot.sh

# 4. Train
bash scripts/run_train.sh 2>&1 | tee output/train.log

# 5. Package adapter
bash scripts/package_submission.sh output/adapter_YYYYMMDD_HHMMSS
```


## 10. Acknowledgements

- **NVIDIA** for the Nemotron-3-Nano-30B model and the competition
- **Kaggle** for hosting the competition and infrastructure
- **Hugging Face** for PEFT, TRL, and Transformers
- **kienngx** for the peer CoT training dataset (Gemini-2.0-flash generated traces)
- **Google DeepMind** for Gemini-2.0-flash used to generate the CoT traces
